# The Lowest RMSE Is Not the Best Decision

Executable companion to [the long-form article](article.md). All data are synthetic and reproduce two different tasks: unconstrained, asymmetric single-product newsvendor decisions and nonanticipative, two-product capacity-constrained stochastic linear programming. For the publication plots and ten CSV exports run [analysis.py](analysis.py).

## 1. Prediction error and downstream loss optimize different objectives

An accurate fitted conditional mean minimizes expected square error under the stated lognormal model. When shortage costs 8 and excess costs 2, the newsvendor solution instead uses conditional quantile 8/(8+2)=0.8. These predictions are used directly as order quantities in the first experiment; all reported losses are holdout losses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import CAPACITY, joint_losses, run_study

study = run_study(quick=True)
for key in ('mean_rmse', 'quantile_rmse', 'mean_decision_cost', 'quantile_decision_cost'):
    print(key, round(study.single_summary[key], 3))
assert study.single_summary['mean_rmse'] < study.single_summary['quantile_rmse']
assert study.single_summary['mean_decision_cost'] > study.single_summary['quantile_decision_cost']

## 2. The critical fractile and the decision loss surface

The optimum minimizes E[c_u(D-q)+ + c_o(q-D)+] and equals conditional demand quantile c_u/(c_u+c_o) under a continuous demand law. The oracle is available only because the synthetic population is defined in code.

In [ ]:
surface = study.loss_surface
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.plot([r['quantity'] for r in surface], [r['mean_cost'] for r in surface])
ax.axvline(study.summary['fixed_x_mean_decision'], ls=':', color='tab:blue', label='Fitted mean')
ax.axvline(study.summary['fixed_x_quantile_decision'], ls='--', color='tab:orange', label='Fitted quantile')
ax.set(xlabel='Order quantity at X = 0.35', ylabel='Expected asymmetric loss')
ax.legend()
plt.show()

## 3. Sample-average approximation (SAA)

The nonparametric SAA decision is an empirical critical-fractile quantile of demand samples. Repeated independently simulated scenario sets show how realized decision regret varies with the scenario count. Regret is evaluated against the known generating-model quantile on a separate test sample and is an estimate, not a guaranteed monotone curve.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.plot([r['scenarios'] for r in study.convergence], [r['mean_decision_regret'] for r in study.convergence], 'o-')
ax.axhline(0, color='black', ls='--')
ax.set(xscale='log', xlabel='Calibration scenarios', ylabel='Estimated cost regret')
plt.show()

## 4. Nonanticipativity, shared capacity, and tail risk

The scenario LP chooses a single two-product allocation before the demand scenarios are realized and enforces q1+q2<=190. Slack variables represent shortages/excess in each scenario. For positive risk weight it includes the Rockafellar-Uryasev CVaR epigraph. The deterministic mean baseline is feasible but leaves capacity unused.

In [ ]:
for name, allocation in study.policies.items():
    print(name, np.round(allocation, 2), 'capacity used:', round(float(allocation.sum()), 1))
    assert float(allocation.sum()) <= CAPACITY + 1e-6
    assert np.all(allocation >= 0)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
risk = study.risk_tradeoff
ax.plot([row['mean_cost'] for row in risk], [row['cvar90'] for row in risk], 'o-')
for row in risk:
    ax.annotate(f"{row['weight_cvar']:.2f}", (row['mean_cost'], row['cvar90']))
ax.set(xlabel='Out-of-sample expected loss', ylabel='Out-of-sample CVaR 90%')
plt.show()

## 5. What information could change the decision?

Expected value of perfect information (EVPI) compares a fixed nonanticipative allocation to a hypothetical decision made after seeing demand. That latter action cannot be deployed. Estimated value of the stochastic solution (VSS) compares a mean-based deterministic decision with the SAA policy, both tested on the same independent demand outcomes. Shifting volatility or dependence changes observed costs but the frozen policies are not refit.

In [ ]:
for row in study.information:
    print(row['scenario'], 'VSS:', round(row['observed_vss'], 2), 'EVPI:', round(row['observed_evpi'], 2))
assert study.summary['estimated_vss'] > 0
assert study.summary['estimated_evpi'] > 0
baseline = study.baseline_joint
for name in ('Deterministic mean', 'SAA expected cost', 'Mean/CVaR 1.00'):
    print(name, 'mean loss:', round(float(joint_losses(study.policies[name], baseline).mean()), 2))